# 02_preprocessing.ipynb

This notebook performs data cleaning and preprocessing for the Cardiovascular Disease dataset.
Place `cardio_train.csv` in the `data/` folder (one level above this `notebooks/` folder).

In [1]:
# Import libraries
import os
import pandas as pd
import numpy as np

# Paths
# Try several common locations for the dataset so the notebook is robust to cwd differences
possible_paths = [
    os.path.join('data', 'cardio_train.csv'),
    os.path.join('..', 'data', 'cardio_train.csv'),
    os.path.join('notebooks', '..', 'data', 'cardio_train.csv'),
    os.path.join('.', 'data', 'cardio_train.csv'),
]

dataset_path = None
for p in possible_paths:
    if os.path.exists(p):
        dataset_path = p
        break
# Default to project-level data folder if none found (will raise below if missing)
if dataset_path is None:
    dataset_path = os.path.join('data', 'cardio_train.csv')

processed_dir = os.path.join('data', 'processed')
os.makedirs(processed_dir, exist_ok=True)
save_path = os.path.join(processed_dir, 'cleaned_data.csv')

# Load dataset with friendly error if missing
try:
    df = pd.read_csv(dataset_path)
    print(f'Loaded dataset from: {os.path.abspath(dataset_path)} with shape: {df.shape}')
except FileNotFoundError:
    raise FileNotFoundError(f'Could not find cardio_train.csv. Please place it at: {os.path.abspath(os.path.join("data","cardio_train.csv"))} or at one of: {possible_paths}')


Loaded dataset from: C:\Users\Noothan\Documents\my_work\HeartDiseaseAI\data\cardio_train.csv with shape: (70000, 13)


## 1. Remove `id` column if present
Removing identifier columns prevents the model from learning spurious patterns.

In [2]:
if 'id' in df.columns:
    df = df.drop(columns=['id'])
    print('Dropped `id` column.')
else:
    print('No `id` column found.')

Dropped `id` column.


## 2. Convert `age` from days to years
We replace the `age` column with age in whole years for readability.

In [3]:
if 'age' in df.columns:
    # Create `age_years` from days and keep `age` as years for readability
    df['age_years'] = (df['age'] / 365).astype(int)
    # Also replace `age` with years to maintain consistency in downstream steps
    df['age'] = df['age_years']
    print('Converted `age` to years and created `age_years`.')
else:
    print('No `age` column found.')


Converted `age` to years and created `age_years`.


## 3. Create BMI feature using `height` and `weight`
BMI = weight (kg) / (height (m))^2. Many datasets provide height in cm, so we convert to meters.

In [4]:
if 'height' in df.columns and 'weight' in df.columns:
    # Convert height from cm to meters if values look like centimeters (> 3 meters would be unrealistic)
    height_mean = df['height'].mean()
    if height_mean > 3:  # likely in cm
        df['height_m'] = df['height'] / 100.0
    else:
        df['height_m'] = df['height']
    df['BMI'] = df['weight'] / (df['height_m'] ** 2)
    df['BMI'] = df['BMI'].round(2)
    print('Created `BMI` feature.')
else:
    print('Height and/or weight columns not found; cannot compute BMI.')

# Create pulse pressure feature from systolic and diastolic blood pressure
if 'ap_hi' in df.columns and 'ap_lo' in df.columns:
    df['pulse_pressure'] = df['ap_hi'] - df['ap_lo']
    print('Created `pulse_pressure` feature.')
else:
    print('ap_hi or ap_lo missing; skipped pulse pressure creation.')

Created `BMI` feature.
Created `pulse_pressure` feature.


## 4–7. Check unrealistic values and handle outliers
We define reasonable thresholds and remove rows that are clearly invalid. Then we handle BMI outliers using the IQR method.

In [5]:
initial_rows = df.shape[0]
# Define masks for unrealistic values (adjust thresholds as needed)
masks = []
# Blood pressure: systolic (ap_hi) and diastolic (ap_lo)
# Systolic must be >= diastolic (medical impossibility check)
if 'ap_hi' in df.columns and 'ap_lo' in df.columns:
    mask_bp_valid = df['ap_hi'] >= df['ap_lo']
    masks.append(mask_bp_valid)
    
# Systolic range: 70-250 mmHg
if 'ap_hi' in df.columns:
    mask_ap_hi = (df['ap_hi'] >= 70) & (df['ap_hi'] <= 250)
    masks.append(mask_ap_hi)
else:
    mask_ap_hi = pd.Series([True] * df.shape[0])
    
# Diastolic range: 40-200 mmHg
if 'ap_lo' in df.columns:
    mask_ap_lo = (df['ap_lo'] >= 40) & (df['ap_lo'] <= 200)
    masks.append(mask_ap_lo)
else:
    mask_ap_lo = pd.Series([True] * df.shape[0])
    
# Height (cm): realistic 100cm - 250cm
if 'height' in df.columns:
    mask_height = (df['height'] >= 100) & (df['height'] <= 250)
    masks.append(mask_height)
else:
    mask_height = pd.Series([True] * df.shape[0])
    
# Weight (kg): realistic 30kg - 300kg
if 'weight' in df.columns:
    mask_weight = (df['weight'] >= 30) & (df['weight'] <= 300)
    masks.append(mask_weight)
else:
    mask_weight = pd.Series([True] * df.shape[0])
    
# Combine masks: keep rows where all conditions are True
if masks:
    combined_mask = np.logical_and.reduce(masks)
    df_clean = df.loc[combined_mask].copy()
else:
    df_clean = df.copy()
    
removed_unrealistic = initial_rows - df_clean.shape[0]
print(f'Removed {removed_unrealistic} rows with clearly unrealistic values.')
print(f'Data shape after removing unrealistic records: {df_clean.shape}')

# Handle BMI outliers using IQR if BMI exists
if 'BMI' in df_clean.columns:
    Q1 = df_clean['BMI'].quantile(0.25)
    Q3 = df_clean['BMI'].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    before_bmi_rows = df_clean.shape[0]
    df_clean = df_clean[(df_clean['BMI'] >= lower) & (df_clean['BMI'] <= upper)].copy()
    removed_bmi = before_bmi_rows - df_clean.shape[0]
    print(f'Removed {removed_bmi} BMI outliers using IQR method.')
else:
    print('BMI not present, skipped BMI outlier removal.')


Removed 1359 rows with clearly unrealistic values.
Data shape after removing unrealistic records: (68641, 16)
Removed 1994 BMI outliers using IQR method.


## 8. Verify data types
Ensure common columns have expected types (integers for counts/ages, floats for BMI).

In [6]:
# Convert/confirm types where reasonable
# Age should be integer years
if 'age' in df_clean.columns:
    df_clean['age'] = df_clean['age'].astype(int)
# BMI should be float
if 'BMI' in df_clean.columns:
    df_clean['BMI'] = df_clean['BMI'].astype(float)
# Print dtypes for verification
print(df_clean.dtypes)

age                 int32
gender              int64
height              int64
weight            float64
ap_hi               int64
ap_lo               int64
cholesterol         int64
gluc                int64
smoke               int64
alco                int64
active              int64
cardio              int64
age_years           int32
height_m          float64
BMI               float64
pulse_pressure      int64
dtype: object


## 9. Save cleaned dataset to `data/processed/cleaned_data.csv`

In [7]:
# Save cleaned dataset
df_clean.to_csv(save_path, index=False)
print(f'Saved cleaned data to: {save_path} (rows: {df_clean.shape[0]}, columns: {df_clean.shape[1]})')

Saved cleaned data to: data\processed\cleaned_data.csv (rows: 66647, columns: 16)


## 10. Display first 5 rows of cleaned dataset

In [8]:
# Show first 5 rows
df_clean.head()

,age,gender,height,weight,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,cardio,age_years,height_m,BMI,pulse_pressure
0,50,2,168,62.0,110,80,1,1,0,0,1,0,50,1.68,21.97,30
1,55,1,156,85.0,140,90,3,1,0,0,1,1,55,1.56,34.93,50
2,51,1,165,64.0,130,70,3,1,0,0,0,1,51,1.65,23.51,60
3,48,2,169,82.0,150,100,1,1,0,0,1,1,48,1.69,28.71,50
4,47,1,156,56.0,100,60,1,1,0,0,0,0,47,1.56,23.01,40
